In [1]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
PRED = OUT + '/predictions'
import os, sys, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
os.makedirs(PRED, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
assert torch.cuda.is_available(), 'Switch runtime to GPU'
dev = 'cuda'
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def load(f):
    return ((torch.tensor(np.load(f'{DATA}/data/{f}')).permute(0, 3, 1, 2).float().div(255) - mean) / std).to(dev)
X, XT = load('train_images.npy'), load('test_images.npy')
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
va = np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)
Xtr, Xva = X[torch.tensor(tr).to(dev)], X[torch.tensor(va).to(dev)]
del X
ys_np = lab.synthetic_label.values[tr]
ys = torch.tensor(ys_np).to(dev)
yvs = torch.tensor(lab.synthetic_label.values[va]).to(dev)
yva_c = torch.tensor(lab.clean_label.values[va]).to(dev)

def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def probs(T):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([F.softmax(net(T[k:k + 1000]).float(), 1) for k in range(0, len(T), 1000)])

Mounted at /content/drive


In [2]:
def train(fname, labels, w=None, pick='best', keep_probs=False, seed=0, E=60, B=128):
    global net
    torch.manual_seed(seed); np.random.seed(seed)
    net = ResNet18(10).to(dev)
    opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
    scaler = torch.amp.GradScaler('cuda')
    Pavg = torch.zeros(len(Xtr), 10, device=dev)
    best, best_ep, best_pred = -1, 0, None
    for ep in range(E):
        net.train()
        perm = torch.randperm(len(Xtr), device=dev)
        for s in range(0, len(perm), B):
            b = perm[s:s + B]
            with torch.autocast('cuda', dtype=torch.float16):
                out = net(augment(Xtr[b]))
            l = F.cross_entropy(out.float(), labels[b], reduction='none')
            loss = l.mean() if w is None else (l * w[b]).sum() / w[b].sum().clamp(min=1e-6)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        if keep_probs:
            Pavg += probs(Xtr) / E
        pv = probs(Xva).argmax(1)
        vs, vc = (pv == yvs).float().mean().item(), (pv == yva_c).float().mean().item()
        if (pick == 'best' and vs > best) or (pick == 'last' and ep == E - 1):
            best, best_ep, best_pred = vs, ep + 1, probs(XT).argmax(1).cpu().numpy()
        print(ep + 1, f'val_synth {vs:.4f}  val_clean {vc:.4f}')
    pd.DataFrame({'id': np.arange(len(XT)), 'pred': best_pred}).to_csv(f'{PRED}/{fname}', index=False)
    print(f'saved {fname}  epoch {best_ep}  val_synth {best:.4f}')
    return round(best, 4), Pavg.cpu().numpy()

In [3]:
base, Ps = train('t5_baseline_synth.csv', ys, pick='last', keep_probs=True)
np.save(f'{OUT}/t5_baseline_train_probs.npy', Ps)

1 val_synth 0.2304  val_clean 0.3488
2 val_synth 0.2780  val_clean 0.4150
3 val_synth 0.2858  val_clean 0.4468
4 val_synth 0.3184  val_clean 0.4924
5 val_synth 0.3508  val_clean 0.5546
6 val_synth 0.3486  val_clean 0.5518
7 val_synth 0.3636  val_clean 0.5794
8 val_synth 0.3646  val_clean 0.5728
9 val_synth 0.3646  val_clean 0.5822
10 val_synth 0.3360  val_clean 0.5346
11 val_synth 0.3854  val_clean 0.6278
12 val_synth 0.3662  val_clean 0.5926
13 val_synth 0.4166  val_clean 0.6790
14 val_synth 0.4216  val_clean 0.6960
15 val_synth 0.4084  val_clean 0.6672
16 val_synth 0.4216  val_clean 0.6874
17 val_synth 0.4074  val_clean 0.6632
18 val_synth 0.4238  val_clean 0.6958
19 val_synth 0.4090  val_clean 0.6550
20 val_synth 0.4586  val_clean 0.7578
21 val_synth 0.4312  val_clean 0.7042
22 val_synth 0.4372  val_clean 0.7134
23 val_synth 0.4552  val_clean 0.7506
24 val_synth 0.4436  val_clean 0.7228
25 val_synth 0.4812  val_clean 0.7950
26 val_synth 0.4592  val_clean 0.7604
27 val_synth 0.4486  

In [4]:
pred, conf = Ps.argmax(1), Ps.max(1)
keep = pred == ys_np
relabel = ~keep & (conf >= 0.5)
drop = ~keep & ~relabel
y_new = ys_np.copy(); y_new[relabel] = pred[relabel]
w = torch.tensor((~drop).astype(np.float32), device=dev)
print(f'keep {keep.sum()}  relabel {relabel.sum()}  drop {drop.sum()}')
clean_tr = lab.clean_label.values[tr]
print('label accuracy, report only:', (ys_np == clean_tr).mean().round(4), '->', (y_new == clean_tr)[~drop].mean().round(4))

keep 28268  relabel 4281  drop 12451
label accuracy, report only: 0.599 -> 0.9401


In [5]:
best_m, _ = train('t5_best_synth.csv', torch.tensor(y_new).to(dev), w=w)

1 val_synth 0.2770  val_clean 0.4248
2 val_synth 0.3296  val_clean 0.5162
3 val_synth 0.3616  val_clean 0.5906
4 val_synth 0.3970  val_clean 0.6402
5 val_synth 0.3972  val_clean 0.6416
6 val_synth 0.3766  val_clean 0.6112
7 val_synth 0.4180  val_clean 0.6764
8 val_synth 0.4628  val_clean 0.7522
9 val_synth 0.4364  val_clean 0.7052
10 val_synth 0.4190  val_clean 0.6842
11 val_synth 0.4440  val_clean 0.7276
12 val_synth 0.4630  val_clean 0.7628
13 val_synth 0.4400  val_clean 0.7204
14 val_synth 0.4580  val_clean 0.7580
15 val_synth 0.4618  val_clean 0.7574
16 val_synth 0.4628  val_clean 0.7590
17 val_synth 0.4454  val_clean 0.7318
18 val_synth 0.4208  val_clean 0.6960
19 val_synth 0.4486  val_clean 0.7438
20 val_synth 0.4658  val_clean 0.7728
21 val_synth 0.4754  val_clean 0.7908
22 val_synth 0.4654  val_clean 0.7658
23 val_synth 0.4916  val_clean 0.8198
24 val_synth 0.4628  val_clean 0.7500
25 val_synth 0.4858  val_clean 0.8102
26 val_synth 0.4962  val_clean 0.8180
27 val_synth 0.4620  

In [6]:
print(json.dumps({'val_acc_synthetic_labels__baseline': base, 'val_acc_synthetic_labels__best_method': best_m}, indent=2))

{
  "val_acc_synthetic_labels__baseline": 0.4268,
  "val_acc_synthetic_labels__best_method": 0.5254
}
